In [ ]:
import numpy as np
import torch
from torchvision import datasets, transforms
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from optimizer import SBOptimizer
from linearsb import LinearSB
import gc
import os

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_dataset = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset  = datasets.MNIST(root='./data', train=False, transform=transform, download=True)

train_loader = DataLoader(dataset=train_dataset, batch_size=16, shuffle=True)
test_loader  = DataLoader(dataset=test_dataset, batch_size=1000, shuffle=False)

In [ ]:
class mnist(nn.Module):
    def __init__(self, num_bases):
        super().__init__()
        q, _ = torch.linalg.qr(torch.randn(784, 784, device=device))
        self.register_buffer('q', q)
        self.layer1 = LinearSB(784, 1024, num_bases=num_bases)
        self.relu1 = nn.ReLU()
        self.drop1 = nn.Dropout(0.2)
        self.layer2 = LinearSB(1024, 1024, num_bases=num_bases)
        self.relu2 = nn.ReLU()
        self.classifier = nn.Linear(1024, 10)
        
    def forward(self, x):
        x = x.flatten(start_dim=1)
        x = x @ self.q
        x = self.relu1(self.layer1(x))
        residual = x
        x = self.drop1(x)
        x = self.relu2(self.layer2(x))
        x = x + residual
        return self.classifier(x)

In [ ]:
grid_num_bases = [1, 2, 3, 5, 10]
grid_num_flips = [1, 2, 3, 5, 10]
grid_results = {}
epochs = 50

In [ ]:
for num_bases in grid_num_bases:
    for num_flips in grid_num_flips:
        print(f"\nTraining with num_bases={num_bases}, num_flips={num_flips}\n")

        model = mnist(num_bases=num_bases).to(device)
        criterion = nn.CrossEntropyLoss()
        sb_params = []
        adam_params = []

        for p in model.parameters():
            if p.requires_grad:
                if getattr(p, 'is_quantized_basis', False):
                    sb_params.append(p)
                else:
                    adam_params.append(p)

        sb_optimizer = SBOptimizer(sb_params, lr=0.01, momentum=0.9, num_flips=num_flips)
        adam_optimizer = optim.Adam(adam_params, lr=0.01)
        gc.collect()

        train_losses = []
        train_accuracies = []
        test_losses = []
        test_accuracies = []
        start_epoch = 0

        checkpoint_path = f"mnist_bases{num_bases}_flips{num_flips}.pt"
        if os.path.exists(checkpoint_path):
            print(f"Found checkpoint '{checkpoint_path}', resuming training")
            checkpoint = torch.load(checkpoint_path, map_location=device)
            model.load_state_dict(checkpoint['model_state_dict'])
            sb_optimizer.load_state_dict(checkpoint['sb_optimizer_state_dict'])
            adam_optimizer.load_state_dict(checkpoint['adam_optimizer_state_dict'])
            start_epoch = checkpoint['epoch'] + 1
            train_losses = checkpoint['train_losses']
            train_accuracies = checkpoint['train_accuracies']
            test_losses = checkpoint['test_losses']
            test_accuracies = checkpoint['test_accuracies']
            print(f"Resuming from epoch {start_epoch + 1}")
        else:
            print(f"No checkpoint found, starting training from scratch")

        if start_epoch >= epochs and test_accuracies:
            grid_results[f"bases_{num_bases}_flips_{num_flips}"] = {
                "num_bases": num_bases,
                "num_flips": num_flips,
                "best_test_acc": max(test_accuracies),
                "final_test_acc": test_accuracies[-1],
                "history": list(test_accuracies)
            }

        for epoch in range(start_epoch, epochs):
            model.train()
            running_loss = 0.0
            correct = 0
            total = 0

            for inputs, labels in train_loader:
                inputs = inputs.to(device)
                labels = labels.to(device)

                sb_optimizer.zero_grad()
                adam_optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                sb_optimizer.step()
                adam_optimizer.step()
                running_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
            print(f'Epoch [{epoch + 1}/{epochs}]')
            print("Training loss", (running_loss / len(train_loader)))  
            train_losses.append(running_loss / len(train_loader))
            print("Training accuracy", (correct *100/ total),'%')
            train_accuracies.append(correct * 100 / total)

            model.eval()
            running_loss_eval = 0.0
            correct = 0
            total = 0

            with torch.no_grad():
                for inputs, labels in test_loader:
                    inputs = inputs.to(device)
                    labels = labels.to(device)

                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    running_loss_eval += loss.item()
                    _, predicted = torch.max(outputs.data, 1)
                    total += labels.size(0)
                    correct += (predicted == labels).sum().item()

            print("Testing loss", (running_loss_eval / len(test_loader)))
            test_losses.append(running_loss_eval / len(test_loader))
            print("Testing accuracy", (correct*100 / total),'%')
            test_accuracies.append(correct * 100 / total)

            checkpoint_data = {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'sb_optimizer_state_dict': sb_optimizer.state_dict(),
                'adam_optimizer_state_dict': adam_optimizer.state_dict(),
                'train_losses': train_losses,
                'train_accuracies': train_accuracies,
                'test_losses': test_losses,
                'test_accuracies': test_accuracies
            }
            tmp_checkpoint_path = checkpoint_path + ".tmp"
            torch.save(checkpoint_data, tmp_checkpoint_path)
            os.replace(tmp_checkpoint_path, checkpoint_path)
            
            gc.collect()

        if test_accuracies:
            grid_results[f"bases_{num_bases}_flips_{num_flips}"] = {
                "num_bases": num_bases,
                "num_flips": num_flips,
                "best_test_acc": max(test_accuracies),
                "final_test_acc": test_accuracies[-1],
                "history": list(test_accuracies)
            }

In [ ]:
print("Grid search done! Summary:")
best_config = None
best_accuracy = -1.0
for run_config, metrics in grid_results.items():
    print(f"{run_config} | Best Acc: {metrics['best_test_acc']:.2f}% | Final Acc: {metrics['final_test_acc']:.2f}%")
    if metrics['best_test_acc'] > best_accuracy:
        best_accuracy = metrics['best_test_acc']
        best_config = run_config
print(f"\nBest configuration: {best_config} with Best Test Accuracy: {best_accuracy:.2f}%")

In [ ]:
import pandas as pd

data = []
for config, metrics in grid_results.items():
    data.append({
        "config": config,
        "num_bases": metrics["num_bases"],
        "num_flips": metrics["num_flips"],
        "best_test_acc": metrics["best_test_acc"],
        "final_test_acc": metrics["final_test_acc"]
    })

df = pd.DataFrame(data)
df.to_csv("grid_search_matrix.csv", index=False)